# 402 — Cross-Lineage Robustness

## Objective

Assess how the two transcriptomic consensus programs from Notebook 401 behave across TCGA tumor projects and DepMap cell-line lineages. The analyses use the existing consensus scores, program definitions, and orientations. TCGA and DepMap are evaluated separately.

## Analysis

- **Lineage structure.** Use eta-squared to quantify the proportion of consensus-score variance associated with tumor project or cell-line lineage. Summarize score dispersion within groups to show how much variation remains among samples or models from the same context.
- **Within-lineage fidelity.** Measure Pearson and Spearman correlations between each consensus score and its source ICA score in groups with at least 15 samples or models. Cell-line ICA scores are aligned to the orientation defined in Notebook 401.
- **Leave-one-lineage-out influence.** Recalculate global fidelity after excluding each tumor project or cell-line lineage in turn, with consensus weights and scores held fixed.

The results describe where each consensus preserves its source-program signal and which groups exert the greatest influence on that agreement across the cohort.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import pandas as pd

from scipy.stats import pearsonr, spearmanr

from pancancer_epigenetics.utils.paths import Paths

In [2]:
# =============================================================================
# Input and output directories
# =============================================================================

CONSENSUS_PROGRAM_DIR = Paths.consensus_programs
TUMOR_PROGRAM_DIR = Paths.tumor_programs
CELL_LINE_PROGRAM_DIR = Paths.cellline_programs
OUTPUT_DIR = Paths.consensus_programs

In [3]:
# =============================================================================
# Authoritative cross-lineage robustness input paths
# =============================================================================

CONSENSUS_CATALOG_PATH = (
    CONSENSUS_PROGRAM_DIR
    / "401_consensus_transcriptomic_program_catalog.csv"
)

TUMOR_CONSENSUS_SCORES_PATH = (
    CONSENSUS_PROGRAM_DIR
    / "401_consensus_tumor_scores.parquet"
)

CELL_LINE_CONSENSUS_SCORES_PATH = (
    CONSENSUS_PROGRAM_DIR
    / "401_consensus_cellline_scores.parquet"
)

TUMOR_NATIVE_SCORES_PATH = (
    TUMOR_PROGRAM_DIR
    / "tcga_primary_tumor_rna_ica_candidate_scores.csv"
)

CELL_LINE_NATIVE_SCORES_PATH = (
    CELL_LINE_PROGRAM_DIR
    / "310_ica_program_scores.parquet"
)

CELL_LINE_COHORT_PATH = (
    Paths.metadata
    / "302_integrated_modeling_cohort.csv"
)

In [4]:
# =============================================================================
# Load authoritative cross-lineage robustness inputs
# =============================================================================

consensus_catalog = pd.read_csv(CONSENSUS_CATALOG_PATH)

tumor_consensus_scores = pd.read_parquet(
    TUMOR_CONSENSUS_SCORES_PATH
)

cell_line_consensus_scores = pd.read_parquet(
    CELL_LINE_CONSENSUS_SCORES_PATH
)

tumor_native_scores = pd.read_csv(
    TUMOR_NATIVE_SCORES_PATH
)

cell_line_native_scores = pd.read_parquet(
    CELL_LINE_NATIVE_SCORES_PATH
)

cell_line_cohort = pd.read_csv(
    CELL_LINE_COHORT_PATH
)


In [5]:
# =============================================================================
# Freeze cross-lineage robustness analysis parameters
# =============================================================================

MIN_LINEAGE_N = 15

# Lineage structure uses eta-squared across all groups. Lineage-specific
# fidelity uses Pearson and Spearman only for groups with n >= MIN_LINEAGE_N.
# LOLO evaluates every group without refitting or reweighting. No resampling,
# p-value gates, or categorical robustness threshold are introduced.

In [6]:
# =============================================================================
# Freeze consensus-program analysis set
# =============================================================================

consensus_programs = (
    consensus_catalog[
        [
            "consensus_program_id",
            "tumor_rna_axis",
            "cell_line_program",
            "orientation_multiplier",
        ]
    ]
    .copy()
)

consensus_program_ids = consensus_programs[
    "consensus_program_id"
].tolist()

consensus_programs

,consensus_program_id,tumor_rna_axis,cell_line_program,orientation_multiplier
0,CONSENSUS_TX_01,RNA_IC150,ICA_PROGRAM_15,1
1,CONSENSUS_TX_02,RNA_IC151,ICA_PROGRAM_47,-1


In [7]:
# =============================================================================
# Construct lineage-aware consensus analysis tables
# =============================================================================

consensus_model_ids = set(cell_line_consensus_scores["ModelID"])
cohort_model_ids = set(cell_line_cohort["ModelID"])
native_model_ids = set(cell_line_native_scores["ModelID"])

if not (consensus_model_ids == cohort_model_ids == native_model_ids):
    raise ValueError(
        "Cell-line ModelID sets differ across Notebooks 302, 310, and 401."
    )

tumor_analysis = (
    tumor_consensus_scores
    .merge(
        tumor_native_scores[
            [
                "case_submitter_id",
                *consensus_programs["tumor_rna_axis"],
            ]
        ],
        on="case_submitter_id",
        how="inner",
        validate="one_to_one",
    )
)

cell_line_analysis = (
    cell_line_consensus_scores
    .merge(
        cell_line_cohort[
            [
                "ModelID",
                "OncotreeLineage",
            ]
        ],
        on="ModelID",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        cell_line_native_scores[
            [
                "ModelID",
                *consensus_programs["cell_line_program"],
            ]
        ],
        on="ModelID",
        how="inner",
        validate="one_to_one",
    )
)

print("Tumor analysis    :", tumor_analysis.shape)
print("Cell-line analysis:", cell_line_analysis.shape)

Tumor analysis    : (9965, 9)
Cell-line analysis: (712, 6)


In [8]:
# =============================================================================
# Define lineage groups eligible for within-lineage analyses
# =============================================================================

tumor_project_counts = tumor_analysis["project_id"].value_counts()

cell_line_lineage_counts = (
    cell_line_analysis["OncotreeLineage"].value_counts()
)

eligible_tumor_projects = tumor_project_counts[
    tumor_project_counts >= MIN_LINEAGE_N
].index.tolist()

eligible_cell_line_lineages = cell_line_lineage_counts[
    cell_line_lineage_counts >= MIN_LINEAGE_N
].index.tolist()

lineage_coverage_summary = pd.DataFrame(
    [
        {
            "system": "tumor",
            "total_groups": len(tumor_project_counts),
            "eligible_groups": len(eligible_tumor_projects),
            "eligible_samples": int(
                tumor_project_counts.loc[eligible_tumor_projects].sum()
            ),
        },
        {
            "system": "cell_line",
            "total_groups": len(cell_line_lineage_counts),
            "eligible_groups": len(eligible_cell_line_lineages),
            "eligible_samples": int(
                cell_line_lineage_counts.loc[
                    eligible_cell_line_lineages
                ].sum()
            ),
        },
    ]
)

lineage_coverage_summary

,system,total_groups,eligible_groups,eligible_samples
0,tumor,33,33,9965
1,cell_line,27,16,642


In [9]:
# =============================================================================
# Define lineage-variance effect-size helper
# =============================================================================

def eta_squared_by_group(data, value_column, group_column):
    """Quantify the fraction of score variance attributable to group structure."""

    values = data[value_column]
    grand_mean = values.mean()

    ss_total = ((values - grand_mean) ** 2).sum()

    group_summary = (
        data
        .groupby(group_column)[value_column]
        .agg(["size", "mean"])
    )

    ss_between = (
        group_summary["size"]
        * (group_summary["mean"] - grand_mean) ** 2
    ).sum()

    return ss_between / ss_total

In [10]:
# =============================================================================
# Quantify lineage structure of frozen consensus scores
# =============================================================================

lineage_eta_squared = []

for consensus_id in consensus_program_ids:
    lineage_eta_squared.extend(
        [
            {
                "system": "tumor",
                "consensus_program_id": consensus_id,
                "lineage_eta_squared": eta_squared_by_group(
                    tumor_analysis,
                    consensus_id,
                    "project_id",
                ),
            },
            {
                "system": "cell_line",
                "consensus_program_id": consensus_id,
                "lineage_eta_squared": eta_squared_by_group(
                    cell_line_analysis,
                    consensus_id,
                    "OncotreeLineage",
                ),
            },
        ]
    )

lineage_eta_squared = pd.DataFrame(lineage_eta_squared)

lineage_eta_squared

,system,consensus_program_id,lineage_eta_squared
0,tumor,CONSENSUS_TX_01,0.458758
1,cell_line,CONSENSUS_TX_01,0.895187
2,tumor,CONSENSUS_TX_02,0.807917
3,cell_line,CONSENSUS_TX_02,0.566491


In [11]:
# =============================================================================
# Define within-lineage score summary helper
# =============================================================================

def summarize_within_lineage(
    data,
    value_column,
    group_column,
    eligible_groups,
):
    """Summarize consensus-score variation within prespecified eligible groups."""

    subset = data.loc[
        data[group_column].isin(eligible_groups),
        [group_column, value_column],
    ]

    summary = (
        subset
        .groupby(group_column)[value_column]
        .agg(
            n="size",
            mean="mean",
            median="median",
            sd="std",
            q25=lambda x: x.quantile(0.25),
            q75=lambda x: x.quantile(0.75),
        )
        .reset_index()
    )

    summary["iqr"] = summary["q75"] - summary["q25"]

    return summary

In [12]:
# =============================================================================
# Summarize within-lineage consensus-score variation
# =============================================================================

within_lineage_summaries = []

for consensus_id in consensus_program_ids:
    tumor_summary = summarize_within_lineage(
        tumor_analysis,
        consensus_id,
        "project_id",
        eligible_tumor_projects,
    ).assign(
        system="tumor",
        consensus_program_id=consensus_id,
    )

    cell_line_summary = summarize_within_lineage(
        cell_line_analysis,
        consensus_id,
        "OncotreeLineage",
        eligible_cell_line_lineages,
    ).assign(
        system="cell_line",
        consensus_program_id=consensus_id,
    )

    within_lineage_summaries.extend(
        [tumor_summary, cell_line_summary]
    )

within_lineage_summary = pd.concat(
    within_lineage_summaries,
    ignore_index=True,
)


In [13]:
# =============================================================================
# Harmonize lineage-group identifier
# =============================================================================

within_lineage_summary["lineage_group"] = (
    within_lineage_summary["project_id"]
    .combine_first(within_lineage_summary["OncotreeLineage"])
)

within_lineage_summary = (
    within_lineage_summary
    .drop(columns=["project_id", "OncotreeLineage"])
    .loc[
        :,
        [
            "system",
            "consensus_program_id",
            "lineage_group",
            "n",
            "mean",
            "median",
            "sd",
            "q25",
            "q75",
            "iqr",
        ],
    ]
)

within_lineage_summary

,system,consensus_program_id,lineage_group,n,mean,median,sd,q25,q75,iqr
0,tumor,CONSENSUS_TX_01,TCGA-ACC,79,-0.527888,-0.606614,0.636166,-1.054111,-0.213992,0.840120
1,tumor,CONSENSUS_TX_01,TCGA-BLCA,405,-0.452190,-0.592119,0.879972,-1.125554,0.048938,1.174492
2,tumor,CONSENSUS_TX_01,TCGA-BRCA,1089,-0.066549,-0.173858,0.771508,-0.598042,0.356977,0.955019
3,tumor,CONSENSUS_TX_01,TCGA-CESC,304,-0.049566,-0.073242,0.849534,-0.647026,0.546099,1.193125
4,tumor,CONSENSUS_TX_01,TCGA-CHOL,35,-0.269676,-0.478122,1.048359,-0.822896,0.050631,0.873527
...,...,...,...,...,...,...,...,...,...,...
93,cell_line,CONSENSUS_TX_02,Ovary/Fallopian Tube,34,0.078346,0.063041,0.553545,-0.422507,0.416744,0.839251
94,cell_line,CONSENSUS_TX_02,Pancreas,28,-0.969786,-1.236278,0.734171,-1.443463,-0.581743,0.861719
95,cell_line,CONSENSUS_TX_02,Peripheral Nervous System,18,1.477459,1.466171,0.164574,1.365446,1.631785,0.266339
96,cell_line,CONSENSUS_TX_02,Skin,37,0.714777,0.906339,0.816837,0.763091,1.013169,0.250079


In [14]:
# =============================================================================
# Summarize within-lineage dispersion across eligible groups
# =============================================================================

within_lineage_dispersion = (
    within_lineage_summary
    .groupby(
        [
            "system",
            "consensus_program_id",
        ]
    )
    .agg(
        eligible_groups=("lineage_group", "size"),
        median_sd=("sd", "median"),
        minimum_sd=("sd", "min"),
        maximum_sd=("sd", "max"),
        median_iqr=("iqr", "median"),
        minimum_iqr=("iqr", "min"),
        maximum_iqr=("iqr", "max"),
    )
    .reset_index()
)

within_lineage_dispersion

,system,consensus_program_id,eligible_groups,median_sd,minimum_sd,maximum_sd,median_iqr,minimum_iqr,maximum_iqr
0,cell_line,CONSENSUS_TX_01,16,0.204965,0.140832,0.732541,0.230439,0.158203,1.317566
1,cell_line,CONSENSUS_TX_02,16,0.567744,0.148014,1.118938,0.585042,0.197609,2.155592
2,tumor,CONSENSUS_TX_01,33,0.701234,0.395650,1.180718,0.896933,0.510484,1.814080
3,tumor,CONSENSUS_TX_02,33,0.322024,0.168076,0.839721,0.439865,0.192011,1.239365


In [15]:
# =============================================================================
# Define lineage-specific native-score fidelity helper
# =============================================================================

def compute_lineage_fidelity(
    data,
    consensus_column,
    native_column,
    group_column,
    eligible_groups,
):
    """Quantify consensus-to-native score fidelity within eligible lineages."""

    results = []

    for group in eligible_groups:
        subset = data.loc[
            data[group_column].eq(group),
            [consensus_column, native_column],
        ]

        pearson_r = pearsonr(
            subset[consensus_column],
            subset[native_column],
        ).statistic

        spearman_r = spearmanr(
            subset[consensus_column],
            subset[native_column],
        ).statistic

        results.append(
            {
                "lineage_group": group,
                "n": len(subset),
                "pearson_r": pearson_r,
                "spearman_r": spearman_r,
            }
        )

    return pd.DataFrame(results)

In [16]:
# =============================================================================
# Align cell-line native scores to frozen consensus orientation
# =============================================================================

for row in consensus_programs.itertuples(index=False):
    oriented_column = f"{row.cell_line_program}_oriented"

    # ICA signs are arbitrary; notebook 400 froze the multiplier that aligns
    # each cell-line program to the tumor-anchored correspondence orientation.
    cell_line_analysis[oriented_column] = (
        cell_line_analysis[row.cell_line_program]
        * row.orientation_multiplier
    )

In [17]:
# =============================================================================
# Compute lineage-specific native-score fidelity
# =============================================================================

lineage_fidelity_results = []

for row in consensus_programs.itertuples(index=False):
    tumor_fidelity = compute_lineage_fidelity(
        tumor_analysis,
        row.consensus_program_id,
        row.tumor_rna_axis,
        "project_id",
        eligible_tumor_projects,
    ).assign(
        system="tumor",
        consensus_program_id=row.consensus_program_id,
    )

    cell_line_fidelity = compute_lineage_fidelity(
        cell_line_analysis,
        row.consensus_program_id,
        f"{row.cell_line_program}_oriented",
        "OncotreeLineage",
        eligible_cell_line_lineages,
    ).assign(
        system="cell_line",
        consensus_program_id=row.consensus_program_id,
    )

    lineage_fidelity_results.extend(
        [tumor_fidelity, cell_line_fidelity]
    )

lineage_fidelity = pd.concat(
    lineage_fidelity_results,
    ignore_index=True,
)

lineage_fidelity

,lineage_group,n,pearson_r,spearman_r,system,consensus_program_id
0,TCGA-BRCA,1089,0.292267,0.258440,tumor,CONSENSUS_TX_01
1,TCGA-UCEC,543,0.445851,0.366113,tumor,CONSENSUS_TX_01
2,TCGA-HNSC,520,0.300955,0.279422,tumor,CONSENSUS_TX_01
3,TCGA-LGG,513,0.218550,0.203375,tumor,CONSENSUS_TX_01
4,TCGA-LUAD,512,0.360614,0.337379,tumor,CONSENSUS_TX_01
...,...,...,...,...,...,...
93,Kidney,19,0.211281,0.321053,cell_line,CONSENSUS_TX_02
94,Peripheral Nervous System,18,0.256169,0.306502,cell_line,CONSENSUS_TX_02
95,Bone,17,0.307398,0.446078,cell_line,CONSENSUS_TX_02
96,Bladder/Urinary Tract,16,0.706878,0.652941,cell_line,CONSENSUS_TX_02


In [18]:
# =============================================================================
# Summarize lineage-specific native-score fidelity
# =============================================================================

lineage_fidelity_summary = (
    lineage_fidelity
    .groupby(
        [
            "system",
            "consensus_program_id",
        ]
    )
    .agg(
        eligible_groups=("lineage_group", "size"),
        median_pearson=("pearson_r", "median"),
        minimum_pearson=("pearson_r", "min"),
        maximum_pearson=("pearson_r", "max"),
        positive_pearson_fraction=(
            "pearson_r",
            lambda x: (x > 0).mean(),
        ),
        median_spearman=("spearman_r", "median"),
        minimum_spearman=("spearman_r", "min"),
        maximum_spearman=("spearman_r", "max"),
        positive_spearman_fraction=(
            "spearman_r",
            lambda x: (x > 0).mean(),
        ),
    )
    .reset_index()
)

lineage_fidelity_summary

,system,consensus_program_id,eligible_groups,median_pearson,minimum_pearson,maximum_pearson,positive_pearson_fraction,median_spearman,minimum_spearman,maximum_spearman,positive_spearman_fraction
0,cell_line,CONSENSUS_TX_01,16,0.248775,-0.028998,0.827359,0.812500,0.259434,-0.037949,0.799349,0.937500
1,cell_line,CONSENSUS_TX_02,16,0.437267,0.153535,0.919265,1.000000,0.442705,-0.104789,0.748376,0.937500
2,tumor,CONSENSUS_TX_01,33,0.315605,-0.076243,0.643098,0.969697,0.282094,-0.135934,0.523074,0.848485
3,tumor,CONSENSUS_TX_02,33,0.272248,0.022895,0.640846,1.000000,0.249491,-0.001242,0.546499,0.969697


In [19]:
# =============================================================================
# Define leave-one-lineage-out fidelity helper
# =============================================================================

def compute_leave_one_lineage_out_fidelity(
    data,
    consensus_column,
    native_column,
    group_column,
):
    """Measure global consensus-to-native fidelity after excluding each lineage."""

    full_pearson = pearsonr(
        data[consensus_column],
        data[native_column],
    ).statistic

    full_spearman = spearmanr(
        data[consensus_column],
        data[native_column],
    ).statistic

    results = []

    for group in data[group_column].drop_duplicates():
        subset = data.loc[data[group_column].ne(group)]

        results.append(
            {
                "excluded_lineage": group,
                "excluded_n": int(data[group_column].eq(group).sum()),
                "pearson_r": pearsonr(
                    subset[consensus_column],
                    subset[native_column],
                ).statistic,
                "spearman_r": spearmanr(
                    subset[consensus_column],
                    subset[native_column],
                ).statistic,
                "full_pearson_r": full_pearson,
                "full_spearman_r": full_spearman,
            }
        )

    return pd.DataFrame(results)

In [20]:
# =============================================================================
# Compute leave-one-lineage-out native-score fidelity
# =============================================================================

leave_one_lineage_out_results = []

for row in consensus_programs.itertuples(index=False):
    tumor_lolo = compute_leave_one_lineage_out_fidelity(
        tumor_analysis,
        row.consensus_program_id,
        row.tumor_rna_axis,
        "project_id",
    ).assign(
        system="tumor",
        consensus_program_id=row.consensus_program_id,
    )

    cell_line_lolo = compute_leave_one_lineage_out_fidelity(
        cell_line_analysis,
        row.consensus_program_id,
        f"{row.cell_line_program}_oriented",
        "OncotreeLineage",
    ).assign(
        system="cell_line",
        consensus_program_id=row.consensus_program_id,
    )

    leave_one_lineage_out_results.extend(
        [tumor_lolo, cell_line_lolo]
    )

leave_one_lineage_out_fidelity = pd.concat(
    leave_one_lineage_out_results,
    ignore_index=True,
)

leave_one_lineage_out_fidelity

,excluded_lineage,excluded_n,pearson_r,spearman_r,full_pearson_r,full_spearman_r,system,consensus_program_id
0,TCGA-GBM,230,0.262910,0.271240,0.262215,0.269040,tumor,CONSENSUS_TX_01
1,TCGA-OV,422,0.256778,0.263611,0.262215,0.269040,tumor,CONSENSUS_TX_01
2,TCGA-LUAD,512,0.262606,0.272340,0.262215,0.269040,tumor,CONSENSUS_TX_01
3,TCGA-LUSC,500,0.260641,0.265613,0.262215,0.269040,tumor,CONSENSUS_TX_01
4,TCGA-PRAD,496,0.265312,0.278313,0.262215,0.269040,tumor,CONSENSUS_TX_01
...,...,...,...,...,...,...,...,...
115,Pleura,7,0.592267,0.351588,0.593238,0.352971,cell_line,CONSENSUS_TX_02
116,Bone,17,0.594539,0.358071,0.593238,0.352971,cell_line,CONSENSUS_TX_02
117,Prostate,7,0.589346,0.352295,0.593238,0.352971,cell_line,CONSENSUS_TX_02
118,Testis,2,0.592278,0.350983,0.593238,0.352971,cell_line,CONSENSUS_TX_02


In [21]:
# =============================================================================
# Quantify leave-one-lineage-out fidelity shifts
# =============================================================================

leave_one_lineage_out_fidelity["pearson_shift"] = (
    leave_one_lineage_out_fidelity["pearson_r"]
    - leave_one_lineage_out_fidelity["full_pearson_r"]
)

leave_one_lineage_out_fidelity["spearman_shift"] = (
    leave_one_lineage_out_fidelity["spearman_r"]
    - leave_one_lineage_out_fidelity["full_spearman_r"]
)

# Absolute shifts quantify influence irrespective of whether exclusion increases
# or decreases the frozen full-cohort consensus-to-native fidelity.
leave_one_lineage_out_fidelity["abs_pearson_shift"] = (
    leave_one_lineage_out_fidelity["pearson_shift"].abs()
)

leave_one_lineage_out_fidelity["abs_spearman_shift"] = (
    leave_one_lineage_out_fidelity["spearman_shift"].abs()
)

leave_one_lineage_out_fidelity

,excluded_lineage,excluded_n,pearson_r,spearman_r,full_pearson_r,full_spearman_r,system,consensus_program_id,pearson_shift,spearman_shift,abs_pearson_shift,abs_spearman_shift
0,TCGA-GBM,230,0.262910,0.271240,0.262215,0.269040,tumor,CONSENSUS_TX_01,0.000694,0.002200,0.000694,0.002200
1,TCGA-OV,422,0.256778,0.263611,0.262215,0.269040,tumor,CONSENSUS_TX_01,-0.005437,-0.005429,0.005437,0.005429
2,TCGA-LUAD,512,0.262606,0.272340,0.262215,0.269040,tumor,CONSENSUS_TX_01,0.000391,0.003300,0.000391,0.003300
3,TCGA-LUSC,500,0.260641,0.265613,0.262215,0.269040,tumor,CONSENSUS_TX_01,-0.001574,-0.003427,0.001574,0.003427
4,TCGA-PRAD,496,0.265312,0.278313,0.262215,0.269040,tumor,CONSENSUS_TX_01,0.003097,0.009273,0.003097,0.009273
...,...,...,...,...,...,...,...,...,...,...,...,...
115,Pleura,7,0.592267,0.351588,0.593238,0.352971,cell_line,CONSENSUS_TX_02,-0.000971,-0.001382,0.000971,0.001382
116,Bone,17,0.594539,0.358071,0.593238,0.352971,cell_line,CONSENSUS_TX_02,0.001301,0.005100,0.001301,0.005100
117,Prostate,7,0.589346,0.352295,0.593238,0.352971,cell_line,CONSENSUS_TX_02,-0.003892,-0.000676,0.003892,0.000676
118,Testis,2,0.592278,0.350983,0.593238,0.352971,cell_line,CONSENSUS_TX_02,-0.000961,-0.001988,0.000961,0.001988


In [22]:
# =============================================================================
# Summarize leave-one-lineage-out influence
# =============================================================================

leave_one_lineage_out_summary = (
    leave_one_lineage_out_fidelity
    .groupby(
        [
            "system",
            "consensus_program_id",
        ]
    )
    .agg(
        full_pearson_r=("full_pearson_r", "first"),
        maximum_abs_pearson_shift=("abs_pearson_shift", "max"),
        full_spearman_r=("full_spearman_r", "first"),
        maximum_abs_spearman_shift=("abs_spearman_shift", "max"),
    )
    .reset_index()
)

leave_one_lineage_out_summary

,system,consensus_program_id,full_pearson_r,maximum_abs_pearson_shift,full_spearman_r,maximum_abs_spearman_shift
0,cell_line,CONSENSUS_TX_01,0.652867,0.494476,0.329210,0.067776
1,cell_line,CONSENSUS_TX_02,0.593238,0.125053,0.352971,0.075465
2,tumor,CONSENSUS_TX_01,0.262215,0.031821,0.269040,0.021640
3,tumor,CONSENSUS_TX_02,0.129049,0.013466,0.120994,0.010218


In [23]:
# =============================================================================
# Identify most influential lineage exclusions
# =============================================================================

most_influential_lineages = (
    leave_one_lineage_out_fidelity
    .loc[
        leave_one_lineage_out_fidelity
        .groupby(
            [
                "system",
                "consensus_program_id",
            ]
        )["abs_pearson_shift"]
        .idxmax(),
        [
            "system",
            "consensus_program_id",
            "excluded_lineage",
            "excluded_n",
            "full_pearson_r",
            "pearson_r",
            "pearson_shift",
            "full_spearman_r",
            "spearman_r",
            "spearman_shift",
        ],
    ]
    .reset_index(drop=True)
)

most_influential_lineages

,system,consensus_program_id,excluded_lineage,excluded_n,full_pearson_r,pearson_r,pearson_shift,full_spearman_r,spearman_r,spearman_shift
0,cell_line,CONSENSUS_TX_01,Lymphoid,86,0.652867,0.158391,-0.494476,0.329210,0.261434,-0.067776
1,cell_line,CONSENSUS_TX_02,Head and Neck,26,0.593238,0.468185,-0.125053,0.352971,0.277506,-0.075465
2,tumor,CONSENSUS_TX_01,TCGA-LAML,134,0.262215,0.294036,0.031821,0.269040,0.275439,0.006399
3,tumor,CONSENSUS_TX_02,TCGA-LGG,513,0.129049,0.142515,0.013466,0.120994,0.129218,0.008224


In [24]:
# =============================================================================
# Identify most influential lineage exclusions for Spearman fidelity
# =============================================================================

most_influential_spearman_lineages = (
    leave_one_lineage_out_fidelity
    .loc[
        leave_one_lineage_out_fidelity
        .groupby(
            [
                "system",
                "consensus_program_id",
            ]
        )["abs_spearman_shift"]
        .idxmax(),
        [
            "system",
            "consensus_program_id",
            "excluded_lineage",
            "excluded_n",
            "full_spearman_r",
            "spearman_r",
            "spearman_shift",
        ],
    ]
    .reset_index(drop=True)
)

most_influential_spearman_lineages

,system,consensus_program_id,excluded_lineage,excluded_n,full_spearman_r,spearman_r,spearman_shift
0,cell_line,CONSENSUS_TX_01,Lymphoid,86,0.329210,0.261434,-0.067776
1,cell_line,CONSENSUS_TX_02,Head and Neck,26,0.352971,0.277506,-0.075465
2,tumor,CONSENSUS_TX_01,TCGA-THCA,505,0.269040,0.247400,-0.021640
3,tumor,CONSENSUS_TX_02,TCGA-UCEC,543,0.120994,0.110777,-0.010218


In [25]:
# =============================================================================
# Integrate cross-lineage robustness evidence
# =============================================================================

# Preserve each robustness dimension separately rather than collapsing
# heterogeneous evidence into a post hoc composite score.
cross_lineage_robustness_summary = (
    lineage_eta_squared
    .merge(
        within_lineage_dispersion,
        on=["system", "consensus_program_id"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        lineage_fidelity_summary.drop(columns="eligible_groups"),
        on=["system", "consensus_program_id"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        leave_one_lineage_out_summary,
        on=["system", "consensus_program_id"],
        how="left",
        validate="one_to_one",
    )
)

cross_lineage_robustness_summary

,system,consensus_program_id,lineage_eta_squared,eligible_groups,median_sd,minimum_sd,maximum_sd,median_iqr,minimum_iqr,maximum_iqr,...,maximum_pearson,positive_pearson_fraction,median_spearman,minimum_spearman,maximum_spearman,positive_spearman_fraction,full_pearson_r,maximum_abs_pearson_shift,full_spearman_r,maximum_abs_spearman_shift
0,tumor,CONSENSUS_TX_01,0.458758,33,0.701234,0.395650,1.180718,0.896933,0.510484,1.814080,...,0.643098,0.969697,0.282094,-0.135934,0.523074,0.848485,0.262215,0.031821,0.269040,0.021640
1,cell_line,CONSENSUS_TX_01,0.895187,16,0.204965,0.140832,0.732541,0.230439,0.158203,1.317566,...,0.827359,0.812500,0.259434,-0.037949,0.799349,0.937500,0.652867,0.494476,0.329210,0.067776
2,tumor,CONSENSUS_TX_02,0.807917,33,0.322024,0.168076,0.839721,0.439865,0.192011,1.239365,...,0.640846,1.000000,0.249491,-0.001242,0.546499,0.969697,0.129049,0.013466,0.120994,0.010218
3,cell_line,CONSENSUS_TX_02,0.566491,16,0.567744,0.148014,1.118938,0.585042,0.197609,2.155592,...,0.919265,1.000000,0.442705,-0.104789,0.748376,0.937500,0.593238,0.125053,0.352971,0.075465


In [26]:
# =============================================================================
# Attach most influential lineage identities
# =============================================================================

pearson_influence = (
    most_influential_lineages[
        [
            "system",
            "consensus_program_id",
            "excluded_lineage",
        ]
    ]
    .rename(
        columns={
            "excluded_lineage": "max_pearson_influence_lineage",
        }
    )
)

spearman_influence = (
    most_influential_spearman_lineages[
        [
            "system",
            "consensus_program_id",
            "excluded_lineage",
        ]
    ]
    .rename(
        columns={
            "excluded_lineage": "max_spearman_influence_lineage",
        }
    )
)

cross_lineage_robustness_summary = (
    cross_lineage_robustness_summary
    .merge(
        pearson_influence,
        on=["system", "consensus_program_id"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        spearman_influence,
        on=["system", "consensus_program_id"],
        how="left",
        validate="one_to_one",
    )
)

cross_lineage_robustness_summary[
    [
        "system",
        "consensus_program_id",
        "max_pearson_influence_lineage",
        "maximum_abs_pearson_shift",
        "max_spearman_influence_lineage",
        "maximum_abs_spearman_shift",
    ]
]

,system,consensus_program_id,max_pearson_influence_lineage,maximum_abs_pearson_shift,max_spearman_influence_lineage,maximum_abs_spearman_shift
0,tumor,CONSENSUS_TX_01,TCGA-LAML,0.031821,TCGA-THCA,0.021640
1,cell_line,CONSENSUS_TX_01,Lymphoid,0.494476,Lymphoid,0.067776
2,tumor,CONSENSUS_TX_02,TCGA-LGG,0.013466,TCGA-UCEC,0.010218
3,cell_line,CONSENSUS_TX_02,Head and Neck,0.125053,Head and Neck,0.075465


In [27]:
# =============================================================================
# Contextualize maximum Pearson influence with within-lineage fidelity
# =============================================================================

# This post hoc diagnostic contextualizes the lineage identified by LOLO.
# It does not redefine robustness criteria or alter consensus classification.
influential_lineage_fidelity = (
    most_influential_lineages[
        [
            "system",
            "consensus_program_id",
            "excluded_lineage",
            "excluded_n",
            "pearson_shift",
            "spearman_shift",
        ]
    ]
    .rename(
        columns={
            "excluded_lineage": "lineage_group",
            "excluded_n": "lineage_n",
        }
    )
    .merge(
        lineage_fidelity[
            [
                "system",
                "consensus_program_id",
                "lineage_group",
                "pearson_r",
                "spearman_r",
            ]
        ],
        on=[
            "system",
            "consensus_program_id",
            "lineage_group",
        ],
        how="left",
        validate="one_to_one",
    )
)

influential_lineage_fidelity

,system,consensus_program_id,lineage_group,lineage_n,pearson_shift,spearman_shift,pearson_r,spearman_r
0,cell_line,CONSENSUS_TX_01,Lymphoid,86,-0.494476,-0.067776,0.827359,0.799349
1,cell_line,CONSENSUS_TX_02,Head and Neck,26,-0.125053,-0.075465,0.827837,0.748376
2,tumor,CONSENSUS_TX_01,TCGA-LAML,134,0.031821,0.006399,-0.076243,-0.135934
3,tumor,CONSENSUS_TX_02,TCGA-LGG,513,0.013466,0.008224,0.129993,0.147305


In [28]:
# =============================================================================
# Define cross-lineage robustness output path
# =============================================================================

ROBUSTNESS_SUMMARY_PATH = (
    OUTPUT_DIR
    / "402_cross_lineage_robustness_summary.csv"
)

In [29]:
# =============================================================================
# Write cross-lineage robustness summary
# =============================================================================

cross_lineage_robustness_summary.to_csv(
    ROBUSTNESS_SUMMARY_PATH,
    index=False,
)

In [30]:
# =============================================================================
# Verify cross-lineage robustness artifact publication
# =============================================================================

published_artifacts = [
    ROBUSTNESS_SUMMARY_PATH,
]

print(
    "Cross-lineage robustness artifacts published:",
    all(path.exists() for path in published_artifacts),
)
print("Artifacts written:", len(published_artifacts))

Cross-lineage robustness artifacts published: True
Artifacts written: 1


## Cross-lineage robustness summary

Both consensus programs showed lineage structure. Within-group fidelity was evaluated across 33 TCGA projects and 16 DepMap lineages with at least 15 samples or models.

| Program | TCGA project η² | DepMap lineage η² | Median within-project Pearson r | Median within-lineage Pearson r |
|---|---:|---:|---:|---:|
| `CONSENSUS_TX_01` | 0.459 | 0.895 | 0.316 | 0.249 |
| `CONSENSUS_TX_02` | 0.808 | 0.566 | 0.272 | 0.437 |

For `CONSENSUS_TX_01`, omitting Lymphoid cell lines reduced global Pearson fidelity from 0.653 to 0.158. Spearman fidelity fell from 0.329 to 0.261, indicating that this lineage had a larger influence on linear agreement than on rank agreement.

For `CONSENSUS_TX_02`, Pearson fidelity was positive in all eligible tumor projects and cell-line lineages. The largest cell-line influence came from Head and Neck. Omitting that lineage reduced global Pearson fidelity from 0.593 to 0.468.

In TCGA, omitting any single project changed global Pearson fidelity by at most 0.032 for `CONSENSUS_TX_01` and 0.013 for `CONSENSUS_TX_02`. These shifts were small relative to the modest full-cohort correlations of 0.262 and 0.129, respectively.

In [ ]:
# =============================================================================
# Verify and register artifacts
# =============================================================================

from pancancer_epigenetics.utils.artifact_registry import (
    load_artifact_registry,
    refresh_notebook_artifacts,
)

output_objects = {
    ROBUSTNESS_SUMMARY_PATH: cross_lineage_robustness_summary,
}
verified_outputs = {}
for path, expected in output_objects.items():
    if not path.is_file():
        raise FileNotFoundError(path)
    if path.suffix == ".json":
        with path.open(encoding="utf-8") as handle:
            observed = json.load(handle)
        if observed != expected:
            raise ValueError(f"Written metadata differs: {path}")
        verified_outputs[path] = None
    else:
        observed = (
            pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
        )
        if observed.shape != expected.shape or not observed.columns.equals(
            expected.columns
        ):
            raise ValueError(f"Written table structure differs: {path}")
        verified_outputs[path] = observed.shape

registered_count = refresh_notebook_artifacts(
    "notebooks/phase4_consensus_programs/402_cross_lineage_robustness.ipynb",
    verified_outputs,
    require_unchanged=False,
)
load_artifact_registry()
print(f"Registered {registered_count} verified artifacts.")
